# 03 · 自编码器 Autoencoder

> **能力标签**：SH8 + SH9（PyTorch 基础 / PyTorch fundamentals + 变分自编码器 VAE / VAE (the hinge)）

## 目标 Objectives

完成本课后，你应该能够：

1. 理解自编码器（Autoencoder）的**编码器（encoder）→ 瓶颈（bottleneck）→ 解码器（decoder）**结构。
2. 用 `nn.Module` 实现一个自编码器，并在小型表格数据上训练重构任务。
3. 理解**重构损失**（reconstruction loss，如 MSE）的含义——迫使解码器从压缩表示恢复原始输入。
4. 可视化瓶颈（latent space）的二维表示，观察学到的低维结构。
5. 理解自编码器的局限性，以及变分自编码器（VAE）如何克服这些局限——引入 VAE 动机。

> 对应能力：**SH8 + SH9**


## 概念讲解 Concepts

### 1. 自编码器结构 Autoencoder Architecture

```
输入 x  ──→  编码器 Encoder  ──→  瓶颈 z  ──→  解码器 Decoder  ──→  重构 x̂
(高维)        (压缩)           (低维)      (解压)              (高维)
```

- **编码器**（Encoder）：将高维输入 x 映射到低维**隐变量**（latent variable）z。
- **瓶颈**（Bottleneck）：z 维度远小于 x，迫使模型学习最重要的结构。
- **解码器**（Decoder）：从 z 重构原始输入 x̂（尽量让 x̂ ≈ x）。

---

### 2. 训练目标：重构损失 Reconstruction Loss

$$\mathcal{L}_{\text{recon}} = \|x - \hat{x}\|^2 = \text{MSE}(x, \hat{x})$$

网络通过最小化重构误差学习**压缩表示**（compact representation）。
没有任何外部标签——这是一种**无监督学习**（unsupervised learning）。

---

### 3. 瓶颈的作用 Role of the Bottleneck

瓶颈维度（latent dim）决定了压缩程度：

| 瓶颈维度 | 效果 |
|------|------|
| 太大（≈ 输入维度） | 恒等映射，什么也学不到 |
| 太小 | 无法重构，信息损失过大 |
| 适中 | 学到有意义的压缩表示 |

---

### 4. 自编码器的局限性 Limitations

**普通自编码器**（Deterministic AE）的隐空间（latent space）没有规则结构：
- 隐向量 z 是**点**（point），没有概率分布约束
- 无法从隐空间**采样**新样本（生成能力弱）
- 相邻的 z 不保证对应相似的输入

**变分自编码器**（VAE，下一节）通过引入正则化解决这些问题：
- 编码器输出 z 的**分布参数**（μ, log σ²），而非固定点
- 用 KL 散度约束 z 接近标准正态分布
- 隐空间结构规整，可以采样生成新样本


## 示例 Worked Example — 在合成表格数据上训练自编码器

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

# ── 合成 tabular 数据（无网络请求）────────────────────────────────────────
def make_data(seed=42):
    torch.manual_seed(seed)
    n_per_class = 50
    centers = torch.tensor([
        [ 2.0,  0.0,  1.0,  0.5, -1.0,  0.0,  0.5,  1.0],
        [-2.0,  1.0, -1.0,  0.5,  1.0,  0.5, -0.5, -1.0],
        [ 0.0, -2.0,  0.5, -1.0,  0.0,  1.5,  1.0, -0.5],
    ])
    X_list, labels_list = [], []
    for c_idx, center in enumerate(centers):
        X_list.append(center + 0.5 * torch.randn(n_per_class, 8))
        labels_list.append(torch.full((n_per_class,), c_idx, dtype=torch.long))
    return torch.cat(X_list), torch.cat(labels_list)

X, labels = make_data()
print(f"数据形状 X.shape = {X.shape}  (样本数=150, 特征数=8)")
print(f"标签类别: {labels.unique().tolist()}")
X_mean, X_std = X.mean(0), X.std(0)
X_norm = (X - X_mean) / (X_std + 1e-8)
print(f"标准化后均值 {X_norm.mean().item():.4f}, 标准差 {X_norm.std().item():.4f}")


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

def make_data(seed=42):
    torch.manual_seed(seed)
    n_per_class = 50
    centers = torch.tensor([
        [ 2.0,  0.0,  1.0,  0.5, -1.0,  0.0,  0.5,  1.0],
        [-2.0,  1.0, -1.0,  0.5,  1.0,  0.5, -0.5, -1.0],
        [ 0.0, -2.0,  0.5, -1.0,  0.0,  1.5,  1.0, -0.5],
    ])
    X_list, labels_list = [], []
    for c_idx, center in enumerate(centers):
        X_list.append(center + 0.5 * torch.randn(n_per_class, 8))
        labels_list.append(torch.full((n_per_class,), c_idx, dtype=torch.long))
    return torch.cat(X_list), torch.cat(labels_list)

X, labels = make_data()
X_mean, X_std = X.mean(0), X.std(0)
X_norm = (X - X_mean) / (X_std + 1e-8)


class Autoencoder(nn.Module):
    # 简单自编码器：8 -> 4 -> 2 -> 4 -> 8
    def __init__(self, in_dim=8, latent_dim=2):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, 4), nn.ReLU(),
            nn.Linear(4, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 4), nn.ReLU(),
            nn.Linear(4, in_dim),
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

    def encode(self, x):
        return self.encoder(x)


torch.manual_seed(0)
ae = Autoencoder(in_dim=8, latent_dim=2)
optimizer = torch.optim.Adam(ae.parameters(), lr=1e-2)
loss_fn   = nn.MSELoss()

n_params = sum(p.numel() for p in ae.parameters())
print(f"自编码器参数总数: {n_params}")

for step in range(200):
    optimizer.zero_grad()
    x_hat = ae(X_norm)
    loss  = loss_fn(x_hat, X_norm)
    loss.backward()
    optimizer.step()
    if step % 50 == 0 or step == 199:
        print(f"  step {step:3d}  reconstruction loss = {loss.item():.6f}")

print()
print(f"最终重构损失 Final recon loss: {loss.item():.6f}  (目标 < 0.5)")
assert loss.item() < 0.5, f"重构损失过高: {loss.item()}"
print("✓ 自编码器训练完成")


In [ ]:
# ── 可视化隐空间 Visualize Latent Space ─────────────────────────────────────
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import tempfile, os
import torch

with torch.no_grad():
    z = ae.encode(X_norm).numpy()

tmpdir = tempfile.mkdtemp()
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

colors = ["#e74c3c", "#3498db", "#2ecc71"]
class_names = ["类别 0", "类别 1", "类别 2"]
for c in range(3):
    mask = labels.numpy() == c
    axes[0].scatter(z[mask, 0], z[mask, 1],
                    c=colors[c], label=class_names[c], alpha=0.7, s=20)
axes[0].set_xlabel("z1 (瓶颈维度 1)")
axes[0].set_ylabel("z2 (瓶颈维度 2)")
axes[0].set_title("自编码器隐空间 AE Latent Space")
axes[0].legend()

with torch.no_grad():
    x_hat_np = ae(X_norm).numpy()
recon_err = ((X_norm.numpy() - x_hat_np) ** 2).mean(axis=1)
axes[1].hist(recon_err, bins=20, color="#9b59b6", alpha=0.8)
axes[1].set_xlabel("每样本重构误差 Per-sample Recon Error")
axes[1].set_ylabel("频数 Count")
axes[1].set_title("重构误差分布 Reconstruction Error Distribution")

fig.tight_layout()
fig.savefig(os.path.join(tmpdir, "ae_latent.png"), dpi=80)
plt.close(fig)

print(f"图像已保存至: {os.path.join(tmpdir, 'ae_latent.png')}")
print()
print("观察 Observations:")
print("  左图：AE 的隐空间将 3 个类别分离，但分布不规整（没有概率约束）。")
print("  右图：大部分样本重构误差很小，证明 AE 成功学到了压缩表示。")
print()
print("AE 局限性：隐空间是任意点，无法从中随机采样生成新样本。")
print("-> 下一节：VAE 通过正则化解决这个问题。")


## 动手 Exercise

In [ ]:
# ── 动手练习：调整瓶颈维度，观察重构质量变化 ─────────────────────────────────
import torch
import torch.nn as nn

torch.manual_seed(42)

def make_data(seed=42):
    torch.manual_seed(seed)
    n_per_class = 50
    centers = torch.tensor([
        [ 2.0,  0.0,  1.0,  0.5, -1.0,  0.0,  0.5,  1.0],
        [-2.0,  1.0, -1.0,  0.5,  1.0,  0.5, -0.5, -1.0],
        [ 0.0, -2.0,  0.5, -1.0,  0.0,  1.5,  1.0, -0.5],
    ])
    X_list = []
    for center in centers:
        X_list.append(center + 0.5 * torch.randn(n_per_class, 8))
    return torch.cat(X_list)

X = make_data()
X_mean, X_std = X.mean(0), X.std(0)
X_norm = (X - X_mean) / (X_std + 1e-8)


class Autoencoder(nn.Module):
    def __init__(self, in_dim=8, latent_dim=2):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(in_dim, 4), nn.ReLU(), nn.Linear(4, latent_dim))
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 4), nn.ReLU(), nn.Linear(4, in_dim))
    def forward(self, x):
        return self.decoder(self.encoder(x))


print(f"{'latent_dim':>12}  {'final_loss':>12}  {'压缩率':>10}")
print("-" * 38)
for ldim in [1, 2, 4]:
    torch.manual_seed(0)
    ae = Autoencoder(in_dim=8, latent_dim=ldim)
    opt = torch.optim.Adam(ae.parameters(), lr=1e-2)
    loss_fn = nn.MSELoss()
    for _ in range(200):
        opt.zero_grad()
        loss = loss_fn(ae(X_norm), X_norm)
        loss.backward()
        opt.step()
    ratio = f"{8}/{ldim} = {8/ldim:.1f}x"
    print(f"{ldim:>12}  {loss.item():>12.6f}  {ratio:>10}")

print()
print("观察：随着 latent_dim 增大，重构损失减小（更多信息被保留）。")
print("但 latent_dim=8 等于输入维度时，模型退化为恒等映射，毫无意义。")


## 延伸阅读 Further Reading

1. **Hinton & Salakhutdinov (2006) «Reducing the Dimensionality of Data with Neural Networks»** — Science — AE 的里程碑论文。
2. **Goodfellow et al. «Deep Learning»** 第 14 章（自编码器）— 系统介绍 AE 的各种变体。
3. **Lilian Weng «From Autoencoder to Beta-VAE»**：<https://lilianweng.github.io/posts/2018-08-12-vae/> — 推荐博文，深入浅出。
4. **PyTorch 官方示例**：<https://github.com/pytorch/examples>


## 关联练习 Related Assignments

自编码器是 VAE 的前置知识，完成本课后请继续学习：

- `04-vae.ipynb`（下一节）— 变分自编码器，引入概率隐空间

然后挑战 VAE 练习：

```bash
python check.py w6-vae
```

> 能力标签：**SH8 + SH9** · threshold ≥ 0.7
